# COMP20008 Assignment 2 — W04G10

**Research Question:** Among Melbourne entire homes and apartments with valid nightly prices, do location and amenities improve high-price prediction beyond property size alone, and which attributes are most useful?

This submission notebook reproduces preprocessing, correlation analysis, KNN/Decision Tree modelling, evaluation, and feature selection from the original Melbourne Inside Airbnb Detailed Listings dataset. It downloads the official 16 June 2026 source directly from Inside Airbnb when `data/listings.csv` is absent.

# Part 1 — Data Preprocessing

## 1. Imports and official data source

In [1]:
from pathlib import Path
import csv
import gzip
import hashlib
import json
import re
import shutil
import ssl
import urllib.request
import urllib.error
from datetime import datetime, timezone

import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
MELBOURNE_SNAPSHOT_DATE = "2026-06-16"
MELBOURNE_LISTINGS_GZ_URL = (
    "https://data.insideairbnb.com/australia/vic/melbourne/"
    "2026-06-16/data/listings.csv.gz"
)
MELBOURNE_LISTINGS_CSV_SHA256 = (
    "526fc94588b7b0656fc147b0a87694267e72878c4f37fc0fb920b6478c524a06"
)

# Works when launched from the repository root or from submission/.
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "submission" else CWD

DISALLOWED_CURATED_COLUMNS = {
    "id", "name", "description", "host_id", "host_identity_verified",
    "host_is_superhost", "host_listings_count", "neighbourhood_cleansed",
    "latitude", "longitude", "property_type", "room_type", "accommodates",
    "bathrooms_text", "bedrooms", "beds", "amenities", "price",
    "minimum_nights", "maximum_nights", "availability_365",
    "has_availability", "number_of_reviews", "first_review", "last_review",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_value", "reviews_per_month",
}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def ensure_melbourne_listings(csv_path):
    """Download and verify the recorded original Melbourne source if absent."""
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    if csv_path.exists():
        actual = sha256_file(csv_path)
        if actual != MELBOURNE_LISTINGS_CSV_SHA256:
            raise ValueError(
                "Existing data/listings.csv is not the recorded Melbourne "
                "16 June 2026 source. Remove it and rerun this notebook. "
                f"SHA256={actual}"
            )
        return csv_path

    gz_path = csv_path.with_suffix(csv_path.suffix + ".gz")
    request = urllib.request.Request(
        MELBOURNE_LISTINGS_GZ_URL,
        headers={"User-Agent": "Mozilla/5.0 COMP20008-W04G10/1.0"},
    )
    ssl_context = ssl.create_default_context(cafile=certifi.where())

    print("Downloading original Inside Airbnb data:")
    print(MELBOURNE_LISTINGS_GZ_URL)
    try:
        with urllib.request.urlopen(
            request, timeout=180, context=ssl_context
        ) as response, gz_path.open("wb") as output:
            shutil.copyfileobj(response, output)
    except (urllib.error.URLError, TimeoutError) as exc:
        gz_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Automatic download failed. Manually download the recorded "
            "Melbourne 16 June 2026 Detailed Listings file, decompress it, "
            f"and place it at {csv_path}."
        ) from exc

    print("Decompressing to:", csv_path)
    with gzip.open(gz_path, "rb") as source, csv_path.open("wb") as output:
        shutil.copyfileobj(source, output)
    gz_path.unlink(missing_ok=True)

    actual = sha256_file(csv_path)
    if actual != MELBOURNE_LISTINGS_CSV_SHA256:
        raise ValueError("Downloaded CSV checksum does not match the recorded source.")

    metadata = {
        "city": "Melbourne",
        "region": "Victoria",
        "country": "Australia",
        "snapshot_date": MELBOURNE_SNAPSHOT_DATE,
        "source_type": "Inside Airbnb Detailed Listings",
        "source_url": MELBOURNE_LISTINGS_GZ_URL,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "csv_sha256": actual,
        "csv_bytes": csv_path.stat().st_size,
    }
    (csv_path.parent / "SOURCE_METADATA.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    return csv_path


def validate_a2_source(frame):
    columns = set(frame.columns)
    if len(frame.columns) == len(DISALLOWED_CURATED_COLUMNS) and columns == DISALLOWED_CURATED_COLUMNS:
        raise ValueError(
            "This file matches the disallowed cleaned teaching dataset. "
            "A2 requires the original Inside Airbnb download."
        )


def validate_melbourne_source(frame):
    required = {"latitude", "longitude", "room_type", "price", "id"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Detailed listings file is missing: {sorted(missing)}")
    lat = pd.to_numeric(frame["latitude"], errors="coerce")
    lon = pd.to_numeric(frame["longitude"], errors="coerce")
    med_lat, med_lon = float(lat.median()), float(lon.median())
    if not (-39.5 <= med_lat <= -36.5 and 143.0 <= med_lon <= 146.5):
        raise ValueError(
            "This file does not appear to be Melbourne data. "
            f"Median coordinates are ({med_lat:.4f}, {med_lon:.4f})."
        )


def parse_price_value(value):
    if pd.isna(value):
        return np.nan
    match = re.search(
        r"^\s*\$?\s*(\d+(?:,\d{3})*)(?:\.(\d+))?\s*$", str(value)
    )
    if not match:
        return np.nan
    integer_part = match.group(1).replace(",", "")
    decimal_part = match.group(2)
    return float(f"{integer_part}.{decimal_part}") if decimal_part else float(integer_part)


def add_clean_price(frame, source="price", target="price_clean"):
    result = frame.copy()
    result[target] = result[source].apply(parse_price_value)
    return result


def parse_amenities(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if text in {"", "[]"}:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
    except (json.JSONDecodeError, TypeError):
        pass
    inner = text[1:-1] if len(text) >= 2 and text[0] in "[{" and text[-1] in "]}" else text
    if not inner:
        return []
    try:
        return [
            item.strip()
            for item in next(csv.reader([inner], skipinitialspace=True))
            if item.strip()
        ]
    except Exception:
        return re.findall(r'"(?:[^"\\]|\\.)*"', text)


def add_amenity_count(frame, source="amenities", target="amenity_count"):
    result = frame.copy()
    result[target] = result[source].apply(lambda value: len(parse_amenities(value)))
    return result


_BATH_NUMBER_RE = re.compile(
    r"^(\d+(?:\.\d+)?)\s+(?:shared\s+|private\s+)?baths?$", re.IGNORECASE
)


def parse_bathrooms_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if "half-bath" in text.casefold():
        return 0.5
    match = _BATH_NUMBER_RE.match(text)
    return float(match.group(1)) if match else np.nan


def add_bathrooms_numeric(frame, source="bathrooms_text", target="bathrooms"):
    result = frame.copy()
    result[target] = result[source].apply(parse_bathrooms_text)
    return result


def normalise_amenity_name(value):
    text = str(value).strip().casefold()
    text = text.replace("\xa0", " ").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text)


CONTROLLED_AMENITY_PATTERNS = {
    "has_pool": re.compile(
        r"^(?:(?:private|shared) )?(?:(?:indoor|outdoor) )?pool(?: -.*)?$"
    ),
    "has_free_parking": re.compile(
        r"^free (?:(?:street|driveway) )?parking(?: (?:garage|lot))?"
        r"(?: on premises)?(?: -.*)?$"
    ),
    "has_dedicated_workspace": re.compile(r"^dedicated workspace$"),
    "has_washer": re.compile(r"^(?:(?:free|paid) )?washer(?: -.*)?$"),
    "has_dryer": re.compile(r"^(?:(?:free|paid) )?dryer(?: -.*)?$"),
}
CONTROLLED_AMENITY_NAMES = {
    "has_air_conditioning": {
        "air conditioning",
        "central air conditioning",
        "portable air conditioning",
        "window ac unit",
    },
}
CONTROLLED_AMENITY_FEATURES = (
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
)


def amenity_indicator(items, feature):
    names = {normalise_amenity_name(item) for item in items}
    if feature in CONTROLLED_AMENITY_NAMES:
        return int(bool(names & CONTROLLED_AMENITY_NAMES[feature]))
    pattern = CONTROLLED_AMENITY_PATTERNS[feature]
    return int(any(pattern.fullmatch(name) for name in names))


def add_amenity_indicators(frame, source="amenities_list"):
    result = frame.copy()
    for feature in CONTROLLED_AMENITY_FEATURES:
        result[feature] = result[source].apply(
            lambda items, selected=feature: amenity_indicator(items, selected)
        )
    return result


DATA_PATH = ensure_melbourne_listings(REPO_ROOT / "data" / "listings.csv")
print("Official source:", MELBOURNE_LISTINGS_GZ_URL)
print("Using:", DATA_PATH.relative_to(REPO_ROOT))

Official source: https://data.insideairbnb.com/australia/vic/melbourne/2026-06-16/data/listings.csv.gz
Using: data/listings.csv


## 2. Load and validate the raw original dataset

The two guards below reject the disallowed 29-column cleaned teaching dataset and a clearly non-Melbourne file.

In [2]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
validate_a2_source(df_raw)
validate_melbourne_source(df_raw)

print("Raw shape:", df_raw.shape)
print("Median coordinates:",
      round(pd.to_numeric(df_raw["latitude"], errors="coerce").median(), 5),
      round(pd.to_numeric(df_raw["longitude"], errors="coerce").median(), 5))
display(df_raw.head())

Raw shape: (25728, 90)
Median coordinates: -37.81739 144.97177


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,10803,https://www.airbnb.com/rooms/10803,20260616211523,2026-06-17,city scrape,"Room in Deco Apartment, Brunswick East",A large air conditioned room with firm queen s...,NaN,https://a0.muscache.com/pictures/e5f30dd1-ac57...,38901,...,4.73,4.70,4.66,NaN,NaN,1,0,1,0,1.31
1,12936,https://www.airbnb.com/rooms/12936,20260616211523,2026-06-28,previous scrape,St Kilda 1BR+BEACHSIDE+BALCONY+WIFI+AC,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,NaN,https://a0.muscache.com/pictures/59701/2e8cdaf...,50121,...,4.83,4.78,4.66,NaN,NaN,10,10,0,0,0.22
2,41836,https://www.airbnb.com/rooms/41836,20260616211523,2026-06-28,previous scrape,CLOSE TO CITY & MELBOURNE AIRPORT,Easy to travel from and to the Airport; quiet ...,NaN,https://a0.muscache.com/pictures/569696dd-1ad0...,182833,...,4.83,4.39,4.69,NaN,NaN,2,0,2,0,0.83
3,43429,https://www.airbnb.com/rooms/43429,20260616211523,2026-06-17,city scrape,Tranquil Javanese Studio and Pond!,"No service/Cleaning Fees, EV Charger, Study th...",NaN,https://a0.muscache.com/pictures/airflow/Hosti...,189684,...,4.94,4.79,4.86,NaN,NaN,2,2,0,0,1.48
4,44699,https://www.airbnb.com/rooms/44699,20260616211523,2026-06-17,city scrape,"15 yearsHosting (4.8), 8 CITY TRAMS, GymPoolTe...",Unwavering service — just ask and we do our be...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,189245,...,4.97,4.84,4.71,NaN,NaN,1,0,1,0,0.35


## 3. Initial audit

In [3]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values(["missing_pct", "missing_n"], ascending=False)

display(audit.head(50))

,dtype,missing_n,missing_pct,n_unique
neighborhood_overview,float64,25728,100.00,0
host_since,float64,25728,100.00,0
host_response_time,float64,25728,100.00,0
host_response_rate,float64,25728,100.00,0
host_acceptance_rate,float64,25728,100.00,0
host_thumbnail_url,float64,25728,100.00,0
host_neighbourhood,float64,25728,100.00,0
host_total_listings_count,float64,25728,100.00,0
host_verifications,float64,25728,100.00,0
neighbourhood,float64,25728,100.00,0


## 4. Define the RQ cohort

The analysis population is restricted to:
- `room_type == "Entire home/apt"`;
- a valid, finite, positive nightly price.

Price is parsed from the original Inside Airbnb currency strings with an anchored numeric rule, as required by the specification.

In [4]:
n_raw = len(df_raw)

df = df_raw.loc[df_raw["room_type"].eq("Entire home/apt")].copy()
n_entire = len(df)

df = add_clean_price(df, source="price", target="price_clean")
valid_price = (
    df["price_clean"].notna()
    & np.isfinite(df["price_clean"])
    & df["price_clean"].gt(0)
)
df = df.loc[valid_price].copy()
n_eligible = len(df)

cohort_summary = pd.DataFrame({
    "stage": [
        "Raw original listings",
        "Entire home/apt",
        "Entire home/apt + valid positive nightly price",
    ],
    "rows": [n_raw, n_entire, n_eligible],
})
cohort_summary["retained_pct_of_raw"] = (
    cohort_summary["rows"] / n_raw * 100
).round(2)

display(cohort_summary)
display(df["price_clean"].describe(percentiles=[.25,.5,.75,.9,.95,.99]))

,stage,rows,retained_pct_of_raw
0,Raw original listings,25728,100.00
1,Entire home/apt,18829,73.18
2,Entire home/apt + valid positive nightly price,14472,56.25


count    14472.000000
mean       368.197452
std        568.704875
min          8.640000
25%        210.655000
50%        280.500000
75%        385.175000
90%        562.000000
95%        759.450000
99%       1756.756900
max      23149.500000
Name: price_clean, dtype: float64

## 5. Six preprocessing candidates

The assignment requires at least six candidates and three selected tasks. The final three map directly to the RQ's three constructs: **property size, location, and amenities**.

Selected:
1. Recompute numeric `bathrooms` from the original `bathrooms_text` field, as required by Section 3.3.
2. Engineer distance from Melbourne CBD from latitude/longitude.
3. Engineer amenity count and a small set of controlled amenity indicators.

Not selected as separate preprocessing tasks:
4. Special missing-value treatment for bedrooms/beds.
5. Property-type consolidation.
6. Neighbourhood encoding/consolidation.

The table below generates the dataset-specific evidence used to justify these choices.


## 6. Selected task 1 — numeric bathrooms from `bathrooms_text`

In [5]:
# Recompute the specification's numeric bathroom variable from the original
# bathrooms_text field. No earlier cleaned dataset is loaded.
raw_numeric_bathrooms = pd.to_numeric(df["bathrooms"], errors="coerce")
raw_numeric_missing_n = int(raw_numeric_bathrooms.isna().sum())

df = add_bathrooms_numeric(
    df,
    source="bathrooms_text",
    target="bathrooms",
)

bathroom_summary = {
    "rows": len(df),
    "raw_numeric_missing_n": raw_numeric_missing_n,
    "raw_numeric_missing_pct": round(raw_numeric_missing_n / len(df) * 100, 2),
    "source_text_categories": int(df["bathrooms_text"].nunique(dropna=True)),
    "source_missing_n": int(df["bathrooms_text"].isna().sum()),
    "numeric_nonmissing_n": int(df["bathrooms"].notna().sum()),
    "numeric_missing_n": int(df["bathrooms"].isna().sum()),
    "numeric_missing_pct": round(float(df["bathrooms"].isna().mean() * 100), 2),
    "numeric_usable_pct": round(float(df["bathrooms"].notna().mean() * 100), 2),
}
bathroom_summary


{'rows': 14472,
 'raw_numeric_missing_n': 1285,
 'raw_numeric_missing_pct': 8.88,
 'source_text_categories': 21,
 'source_missing_n': 1,
 'numeric_nonmissing_n': 14471,
 'numeric_missing_n': 1,
 'numeric_missing_pct': 0.01,
 'numeric_usable_pct': 99.99}

## 7. Selected task 2 — distance from Melbourne CBD

Distance is calculated with the Haversine formula using approximately `(-37.8136, 144.9631)` as the CBD reference point.

In [6]:
def haversine_km(lat, lon, ref_lat=-37.8136, ref_lon=144.9631):
    lat1 = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon1 = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat2 = np.radians(ref_lat)
    lon2 = np.radians(ref_lon)

    dlat = lat1 - lat2
    dlon = lon1 - lon2
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

valid_coordinate_pairs_n = int((
    pd.to_numeric(df["latitude"], errors="coerce").notna()
    & pd.to_numeric(df["longitude"], errors="coerce").notna()
).sum())
df["distance_cbd_km"] = haversine_km(df["latitude"], df["longitude"])
display(df["distance_cbd_km"].describe(percentiles=[.25,.5,.75,.9,.95]))


count    14472.000000
mean        10.354091
std         14.335911
min          0.015709
25%          1.137856
50%          4.019169
75%         13.306550
90%         33.380111
95%         43.711306
max         79.999033
Name: distance_cbd_km, dtype: float64

## 8. Selected task 3 — amenities engineering

Amenities are parsed into individual names before matching. Indicators use exact allowed names or anchored controlled patterns, not substring search. This prevents `Hair dryer`, `Dishwasher`, `Pool table`, `Pool view`, and `Whirlpool` appliance names from being misclassified.

The task creates:
- `amenity_count`;
- six interpretable, controlled amenity indicators for the full prediction model.

`has_free_parking` means the listing advertises a controlled free-parking amenity, including free street parking. It measures advertised access, not ownership or an exclusive on-premises space. `has_dedicated_workspace` replaces the near-constant kitchen indicator.


In [7]:
df = add_amenity_count(df, source="amenities", target="amenity_count")
df["amenities_list"] = df["amenities"].apply(parse_amenities)
df = add_amenity_indicators(df, source="amenities_list")

AMENITY_FEATURES = list(CONTROLLED_AMENITY_FEATURES)
amenity_indicator_summary = pd.DataFrame({
    "feature": AMENITY_FEATURES,
    "n_yes": [int(df[column].sum()) for column in AMENITY_FEATURES],
    "pct_yes": [
        round(float(df[column].mean() * 100), 2)
        for column in AMENITY_FEATURES
    ],
})

amenity_indicator_definitions = pd.DataFrame([
    {
        "feature": "has_free_parking",
        "matching_rule": "anchored controlled free-parking amenity names",
        "scope_note": (
            "Includes advertised free street, driveway, garage, lot, and "
            "on-premises parking; indicates access, not ownership/exclusivity."
        ),
    },
    {
        "feature": "has_dedicated_workspace",
        "matching_rule": "exact normalised name: dedicated workspace",
        "scope_note": "Replaces near-constant kitchen with a more discriminating amenity.",
    },
    {
        "feature": "has_pool",
        "matching_rule": "anchored pool names, optionally private/shared/indoor/outdoor",
        "scope_note": "Excludes pool table, pool view, and whirlpool appliance names.",
    },
    {
        "feature": "has_washer",
        "matching_rule": "anchored washer names, optionally free/paid",
        "scope_note": "Excludes dishwasher and washer appliance descriptions.",
    },
    {
        "feature": "has_dryer",
        "matching_rule": "anchored dryer names, optionally free/paid",
        "scope_note": "Excludes hair dryer and washer-dryer appliance descriptions.",
    },
    {
        "feature": "has_air_conditioning",
        "matching_rule": "exact controlled air-conditioning names",
        "scope_note": "Includes central, portable, and window AC labels.",
    },
])

# Quantify the effect of replacing substring matching.
LEGACY_SUBSTRING_KEYWORDS = {
    "has_pool": ("pool",),
    "has_free_parking": ("free parking",),
    "has_air_conditioning": ("air conditioning", "window ac", "central air"),
    "has_dedicated_workspace": ("dedicated workspace",),
    "has_washer": ("washer",),
    "has_dryer": ("dryer",),
}

amenity_audit_rows = []
for feature, keywords in LEGACY_SUBSTRING_KEYWORDS.items():
    legacy = df["amenities_list"].apply(
        lambda items, selected=keywords: any(
            keyword in " | ".join(str(item).casefold() for item in items)
            for keyword in selected
        )
    )
    controlled = df[feature].astype(bool)
    amenity_audit_rows.append({
        "feature": feature,
        "legacy_substring_n": int(legacy.sum()),
        "controlled_n": int(controlled.sum()),
        "legacy_only_false_positive_n": int((legacy & ~controlled).sum()),
        "controlled_only_new_match_n": int((controlled & ~legacy).sum()),
    })

amenity_matching_audit = pd.DataFrame(amenity_audit_rows)
display(amenity_indicator_summary)
display(amenity_indicator_definitions)
display(amenity_matching_audit)
display(df["amenity_count"].describe(percentiles=[.25, .5, .75, .9, .95]))


,feature,n_yes,pct_yes
0,has_pool,4465,30.85
1,has_free_parking,9643,66.63
2,has_air_conditioning,11455,79.15
3,has_dedicated_workspace,7915,54.69
4,has_washer,13029,90.03
5,has_dryer,8954,61.87


,feature,matching_rule,scope_note
0,has_free_parking,anchored controlled free-parking amenity names,"Includes advertised free street, driveway, gar..."
1,has_dedicated_workspace,exact normalised name: dedicated workspace,Replaces near-constant kitchen with a more dis...
2,has_pool,"anchored pool names, optionally private/shared...","Excludes pool table, pool view, and whirlpool ..."
3,has_washer,"anchored washer names, optionally free/paid",Excludes dishwasher and washer appliance descr...
4,has_dryer,"anchored dryer names, optionally free/paid",Excludes hair dryer and washer-dryer appliance...
5,has_air_conditioning,exact controlled air-conditioning names,"Includes central, portable, and window AC labels."


,feature,legacy_substring_n,controlled_n,legacy_only_false_positive_n,controlled_only_new_match_n
0,has_pool,4610,4465,145,0
1,has_free_parking,7777,9643,0,1866
2,has_air_conditioning,11455,11455,0,0
3,has_dedicated_workspace,7915,7915,0,0
4,has_washer,13333,13029,304,0
5,has_dryer,12850,8954,3896,0


count    14472.000000
mean        41.024323
std         14.544795
min          0.000000
25%         33.000000
50%         43.000000
75%         51.000000
90%         58.000000
95%         63.000000
max         91.000000
Name: amenity_count, dtype: float64

## 9. Dataset-specific candidate decision table

The rejected candidates are still handled where necessary:
- bedrooms/beds missingness is handled inside model pipelines with training-fold median imputation;
- property type is not added because it is outside the RQ's size/location/amenity comparison;
- neighbourhood encoding is not used because distance provides a compact location feature without 30 dummy variables.

In [8]:
property_counts = df["property_type"].value_counts(dropna=False)
neighbour_counts = df["neighbourhood_cleansed"].value_counts(dropna=False)

bedrooms_missing_n = int(df["bedrooms"].isna().sum())
beds_missing_n = int(df["beds"].isna().sum())

candidate_decisions = pd.DataFrame([
    {
        "candidate": "Numeric bathrooms derived from bathrooms_text",
        "selected": True,
        "dataset_evidence": (
            f'{bathroom_summary["numeric_nonmissing_n"]:,}/{len(df):,} rows '
            f'({bathroom_summary["numeric_usable_pct"]:.2f}%) receive a numeric value '
            f'from {bathroom_summary["source_text_categories"]} text categories.'
        ),
        "alternative_considered": "Keep bathrooms_text as a categorical predictor",
        "why_alternative_not_selected": (
            f'bathrooms_text has {bathroom_summary["source_text_categories"]} categories; '
            f'the specified numeric derivation retains {bathroom_summary["numeric_usable_pct"]:.2f}% '
            'coverage and gives a direct property-size measure.'
        ),
    },
    {
        "candidate": "Distance from CBD from latitude/longitude",
        "selected": True,
        "dataset_evidence": (
            f'{int(df["distance_cbd_km"].notna().sum()):,}/{len(df):,} rows receive distance; '
            f'median={df["distance_cbd_km"].median():.2f} km.'
        ),
        "alternative_considered": "One-hot encode neighbourhood_cleansed",
        "why_alternative_not_selected": (
            f'neighbourhood_cleansed has {neighbour_counts.size} categories and '
            f'{int(df["neighbourhood_cleansed"].isna().sum())} missing rows; '
            'distance gives one continuous location feature aligned with the RQ.'
        ),
    },
    {
        "candidate": "Amenity count + amenity indicators",
        "selected": True,
        "dataset_evidence": (
            f'Raw amenities has {df["amenities"].nunique(dropna=True):,} unique strings; '
            f'amenity_count has {df["amenity_count"].nunique(dropna=True)} unique values '
            f'across {len(df):,} rows.'
        ),
        "alternative_considered": "Use the raw amenities string as a categorical feature",
        "why_alternative_not_selected": (
            f'raw amenities contains {df["amenities"].nunique(dropna=True):,} unique strings '
            f'across {len(df):,} eligible rows, so it is close to listing-specific; '
            'count + six interpretable indicators provides a compact representation.'
        ),
    },
    {
        "candidate": "Special bedroom/bed missing-value preprocessing",
        "selected": False,
        "dataset_evidence": (
            f'Bedrooms missing: {bedrooms_missing_n:,} '
            f'({bedrooms_missing_n/len(df)*100:.2f}%); beds missing: {beds_missing_n:,} '
            f'({beds_missing_n/len(df)*100:.2f}%). Median imputation is kept inside model CV.'
        ),
        "alternative_considered": "Standalone pre-imputation before splitting",
        "why_alternative_not_selected": (
            'Imputation is performed inside the modelling pipeline so training-fold '
            'statistics do not leak into validation/test data.'
        ),
    },
    {
        "candidate": "Property-type consolidation",
        "selected": False,
        "dataset_evidence": (
            f'{property_counts.size} property types; '
            f'{int((property_counts < 10).sum())} have fewer than 10 eligible listings.'
        ),
        "alternative_considered": "Collapse rare types to Other",
        "why_alternative_not_selected": (
            'Property type is outside the planned size/location/amenities comparison '
            'and would change the RQ feature-set contrast.'
        ),
    },
    {
        "candidate": "Neighbourhood encoding/consolidation",
        "selected": False,
        "dataset_evidence": (
            f'{neighbour_counts.size} neighbourhood categories with '
            f'{int(df["neighbourhood_cleansed"].isna().sum())} missing rows.'
        ),
        "alternative_considered": "One-hot encode all neighbourhood categories",
        "why_alternative_not_selected": (
            f'This would introduce {neighbour_counts.size} categorical levels; '
            'distance-to-CBD is retained as the primary compact location representation.'
        ),
    },
])

display(candidate_decisions)

selected_task_alternatives = candidate_decisions.loc[
    candidate_decisions["selected"],
    [
        "candidate",
        "dataset_evidence",
        "alternative_considered",
        "why_alternative_not_selected",
    ],
].copy()

display(selected_task_alternatives)

,candidate,selected,dataset_evidence,alternative_considered,why_alternative_not_selected
0,Numeric bathrooms derived from bathrooms_text,True,"14,471/14,472 rows (99.99%) receive a numeric ...",Keep bathrooms_text as a categorical predictor,bathrooms_text has 21 categories; the specifie...
1,Distance from CBD from latitude/longitude,True,"14,472/14,472 rows receive distance; median=4....",One-hot encode neighbourhood_cleansed,neighbourhood_cleansed has 30 categories and 0...
2,Amenity count + amenity indicators,True,"Raw amenities has 13,995 unique strings; ameni...",Use the raw amenities string as a categorical ...,"raw amenities contains 13,995 unique strings a..."
3,Special bedroom/bed missing-value preprocessing,False,Bedrooms missing: 370 (2.56%); beds missing: 5...,Standalone pre-imputation before splitting,Imputation is performed inside the modelling p...
4,Property-type consolidation,False,36 property types; 17 have fewer than 10 eligi...,Collapse rare types to Other,Property type is outside the planned size/loca...
5,Neighbourhood encoding/consolidation,False,30 neighbourhood categories with 0 missing rows.,One-hot encode all neighbourhood categories,This would introduce 30 categorical levels; di...


,candidate,dataset_evidence,alternative_considered,why_alternative_not_selected
0,Numeric bathrooms derived from bathrooms_text,"14,471/14,472 rows (99.99%) receive a numeric ...",Keep bathrooms_text as a categorical predictor,bathrooms_text has 21 categories; the specifie...
1,Distance from CBD from latitude/longitude,"14,472/14,472 rows receive distance; median=4....",One-hot encode neighbourhood_cleansed,neighbourhood_cleansed has 30 categories and 0...
2,Amenity count + amenity indicators,"Raw amenities has 13,995 unique strings; ameni...",Use the raw amenities string as a categorical ...,"raw amenities contains 13,995 unique strings a..."


## 10. Measurable impact of the three selected preprocessing tasks

In [9]:
preprocessing_impact = pd.DataFrame([
    {
        "task": "Numeric bathrooms",
        "before": (
            f'original numeric field missing {bathroom_summary["raw_numeric_missing_n"]:,} '
            f'rows ({bathroom_summary["raw_numeric_missing_pct"]:.2f}%)'
        ),
        "after": (
            f'parsed bathrooms_text missing {bathroom_summary["numeric_missing_n"]:,} '
            f'rows ({bathroom_summary["numeric_missing_pct"]:.2f}%)'
        ),
        "rows_affected": int(
            bathroom_summary["raw_numeric_missing_n"]
            - bathroom_summary["numeric_missing_n"]
        ),
    },
    {
        "task": "Distance from CBD",
        "before": f'{valid_coordinate_pairs_n:,} valid latitude/longitude pairs in two columns',
        "after": (
            f'{int(df["distance_cbd_km"].notna().sum()):,} derived distances; '
            f'median={df["distance_cbd_km"].median():.2f} km, '
            f'IQR={df["distance_cbd_km"].quantile(.25):.2f}–'
            f'{df["distance_cbd_km"].quantile(.75):.2f} km'
        ),
        "rows_affected": int(df["distance_cbd_km"].notna().sum()),
    },
    {
        "task": "Amenities engineering",
        "before": f'{df["amenities"].nunique(dropna=True):,} unique raw amenity strings',
        "after": (
            f'{df["amenity_count"].nunique(dropna=True)} amenity-count values + '
            f'{len(AMENITY_FEATURES)} controlled binary indicators'
        ),
        "rows_affected": int(len(df)),
    },
])

display(preprocessing_impact)


,task,before,after,rows_affected
0,Numeric bathrooms,"original numeric field missing 1,285 rows (8.88%)",parsed bathrooms_text missing 1 rows (0.01%),1284
1,Distance from CBD,"14,472 valid latitude/longitude pairs in two c...","14,472 derived distances; median=4.02 km, IQR=...",14472
2,Amenities engineering,"13,995 unique raw amenity strings",91 amenity-count values + 6 controlled binary ...,14472


## 11. Shared target and split definition

The group contract defines **high price** as price above the **training sample's 75th percentile**.

To also satisfy the rubric's stratified-split requirement without using test prices to set the threshold, the code iterates:
1. stratify using the current threshold;
2. compute Q75 from the resulting training sample only;
3. repeat until the threshold is unchanged.

At convergence, the split is stratified by the same labels produced by the final training-sample threshold.

In [10]:
def make_training_q75_stratified_split(
    frame,
    price_col="price_clean",
    test_size=0.20,
    random_state=42,
    max_iter=50,
    atol=1e-10,
):
    indices = np.arange(len(frame))
    threshold = float(frame[price_col].quantile(0.75))
    history = []

    for iteration in range(1, max_iter + 1):
        labels = (frame[price_col].to_numpy() > threshold).astype(int)

        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            random_state=random_state,
            stratify=labels,
        )

        new_threshold = float(
            frame.iloc[train_idx][price_col].quantile(0.75)
        )
        history.append({
            "iteration": iteration,
            "threshold_used_for_stratification": threshold,
            "training_q75": new_threshold,
        })

        if np.isclose(new_threshold, threshold, rtol=0, atol=atol):
            final_labels = (
                frame[price_col].to_numpy() > new_threshold
            ).astype(int)
            return train_idx, test_idx, new_threshold, final_labels, pd.DataFrame(history)

        threshold = new_threshold

    raise RuntimeError(
        "Training-Q75 / stratified-split iteration did not converge. "
        "Review the split design with the tutor."
    )

train_idx, test_idx, price_threshold, target, split_history = (
    make_training_q75_stratified_split(
        df,
        random_state=RANDOM_STATE,
    )
)

df["high_price"] = target
df["split"] = "test"
df.iloc[train_idx, df.columns.get_loc("split")] = "train"

display(split_history)

split_summary = (
    df.groupby(["split", "high_price"])
      .size()
      .rename("n")
      .reset_index()
)
split_summary["pct_within_split"] = (
    split_summary["n"]
    / split_summary.groupby("split")["n"].transform("sum")
    * 100
).round(2)

print("Final training-sample Q75 threshold:", price_threshold)
display(split_summary)

,iteration,threshold_used_for_stratification,training_q75
0,1,385.175,385.15
1,2,385.150,385.15


Final training-sample Q75 threshold: 385.15


,split,high_price,n,pct_within_split
0,test,0,2171,74.99
1,test,1,724,25.01
2,train,0,8683,75.00
3,train,1,2894,25.00


## 12. Export the reproducible processed dataset and evidence

The development notebooks read this generated file to keep target, split, rows, and feature engineering consistent. The final submission notebook is standalone and creates the same dataframe in memory before continuing.


In [11]:
SIZE_FEATURES = ["accommodates", "bedrooms", "beds", "bathrooms"]
LOCATION_FEATURES = ["distance_cbd_km"]
AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
]

export_cols = [
    "id",
    "price_clean",
    "high_price",
    "split",
    "property_type",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
] + SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES

processed = df[export_cols].copy()

DATA_OUT = REPO_ROOT / "data" / "processed_listings.csv"
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

processed.to_csv(DATA_OUT, index=False)
candidate_decisions.to_csv(TABLE_OUT / "preprocessing_candidates.csv", index=False)
selected_task_alternatives.to_csv(TABLE_OUT / "preprocessing_selected_alternatives.csv", index=False)
preprocessing_impact.to_csv(TABLE_OUT / "preprocessing_impact.csv", index=False)
split_summary.to_csv(TABLE_OUT / "split_summary.csv", index=False)
amenity_indicator_summary.to_csv(TABLE_OUT / "amenity_indicator_prevalence.csv", index=False)
amenity_indicator_definitions.to_csv(TABLE_OUT / "amenity_indicator_definitions.csv", index=False)
amenity_matching_audit.to_csv(
    TABLE_OUT / "amenity_indicator_matching_audit.csv", index=False
)

metadata = {
    "source_url": MELBOURNE_LISTINGS_GZ_URL,
    "raw_rows": int(n_raw),
    "entire_home_rows": int(n_entire),
    "eligible_rows": int(n_eligible),
    "training_price_q75": float(price_threshold),
    "train_rows": int((processed["split"] == "train").sum()),
    "test_rows": int((processed["split"] == "test").sum()),
    "size_features": SIZE_FEATURES,
    "location_features": LOCATION_FEATURES,
    "amenity_features": AMENITY_FEATURES,
}
(TABLE_OUT / "preprocessing_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Saved:", DATA_OUT.relative_to(REPO_ROOT))
print("Processed shape:", processed.shape)
display(processed.head())

Saved: data/processed_listings.csv
Processed shape: (14472, 20)


,id,price_clean,high_price,split,property_type,neighbourhood_cleansed,latitude,longitude,accommodates,bedrooms,beds,bathrooms,distance_cbd_km,amenity_count,has_pool,has_free_parking,has_air_conditioning,has_dedicated_workspace,has_washer,has_dryer
3,43429,165.00,0,train,Entire rental unit,Monash,-37.89983,145.11579,2,1.0,1.0,1.0,16.481410,62,0,1,1,1,1,1
6,51592,294.10,0,test,Entire loft,Melbourne,-37.81266,144.96313,2,1.0,1.0,1.5,0.104557,40,0,0,0,1,1,1
8,2067618,207.50,0,train,Entire townhouse,Whitehorse,-37.82755,145.11753,6,3.0,3.0,2.0,13.653050,42,0,1,1,0,1,0
9,2079904,192.12,0,train,Entire rental unit,Melbourne,-37.81657,144.95325,4,1.0,2.0,1.0,0.926140,51,1,0,1,0,1,0
12,2099179,85.71,0,test,Entire rental unit,Port Phillip,-37.85897,144.99837,2,1.0,1.0,1.0,5.919863,61,0,1,0,1,1,1


## 13. Simple data visualisations for later report selection

In [12]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["price_clean"].clip(upper=df["price_clean"].quantile(0.99)), bins=40)
ax.axvline(price_threshold, linestyle="--", linewidth=1.5)
ax.set_xlabel("Nightly price (AUD; clipped at 99th percentile for display)")
ax.set_ylabel("Listings")
ax.set_title("Eligible Melbourne entire-home nightly prices")
fig.tight_layout()
fig.savefig(FIG_OUT / "eligible_price_distribution.png", dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["distance_cbd_km"], bins=40)
ax.set_xlabel("Distance from Melbourne CBD (km)")
ax.set_ylabel("Listings")
ax.set_title("Distance-to-CBD distribution")
fig.tight_layout()
fig.savefig(FIG_OUT / "distance_cbd_distribution.png", dpi=200)
plt.close(fig)

print("Saved preprocessing figures to:", FIG_OUT.relative_to(REPO_ROOT))

Saved preprocessing figures to: output/figures


# Part 2 — Correlation Analysis

## 1. Imports and in-memory preprocessing result

This section uses `processed`, created above in the same notebook. It does not depend on a development notebook or a pre-existing generated CSV.

In [13]:
from itertools import combinations

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

df = processed.copy()
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target counts:", df["high_price"].value_counts().sort_index().to_dict())

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}
Target counts: {0: 10854, 1: 3618}


## 2. Variable set and method implementation

Pearson and Spearman are computed directly on numeric representations.

For MI/NMI, continuous/count variables are discretised into up to five quantile bins. The binary target is left as binary. This makes the pairwise MI/NMI calculation symmetric and reproducible; the report should explicitly state this implementation choice.

In [14]:
VARIABLES = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "high_price",
]

missing_cols = [c for c in VARIABLES if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing correlation columns: {missing_cols}")

VARIABLE_TYPES = {
    "accommodates": "numeric",
    "bedrooms": "numeric",
    "bathrooms": "numeric",
    "distance_cbd_km": "numeric",
    "amenity_count": "numeric",
    "high_price": "binary",
}

display(df[VARIABLES].describe())

,accommodates,bedrooms,bathrooms,distance_cbd_km,amenity_count,high_price
count,14472.000000,14102.000000,14471.000000,14472.000000,14472.000000,14472.000000
mean,4.554381,2.070132,1.485315,10.354091,41.024323,0.250000
std,2.515616,1.097852,0.742045,14.335911,14.544795,0.433028
min,1.000000,1.000000,0.000000,0.015709,0.000000,0.000000
25%,2.000000,1.000000,1.000000,1.137856,33.000000,0.000000
50%,4.000000,2.000000,1.000000,4.019169,43.000000,0.000000
75%,6.000000,3.000000,2.000000,13.306550,51.000000,0.250000
max,16.000000,12.000000,12.000000,79.999033,91.000000,1.000000


## 3. Helpers for MI/NMI discretisation

In [15]:
def to_information_categories(series, variable_type, max_bins=5):
    if variable_type == "binary":
        return pd.to_numeric(series, errors="coerce")

    numeric = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    valid = numeric.notna()

    if valid.sum() < 2:
        return out

    # qcut may drop duplicate edges for low-cardinality numeric variables.
    binned = pd.qcut(
        numeric.loc[valid],
        q=min(max_bins, numeric.loc[valid].nunique()),
        labels=False,
        duplicates="drop",
    )
    out.loc[valid] = binned.astype(float)
    return out

## 4. Compute all four methods for every pair

In [16]:
rows = []

for a, b in combinations(VARIABLES, 2):
    pair = df[[a, b]].dropna().copy()

    pearson = pearsonr(pair[a], pair[b]).statistic
    spearman = spearmanr(pair[a], pair[b]).statistic

    a_disc = to_information_categories(
        pair[a], VARIABLE_TYPES[a]
    )
    b_disc = to_information_categories(
        pair[b], VARIABLE_TYPES[b]
    )
    valid = a_disc.notna() & b_disc.notna()

    mi = mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )
    nmi = normalized_mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )

    rows.append({
        "var_a": a,
        "var_b": b,
        "n": len(pair),
        "pearson": float(pearson),
        "spearman": float(spearman),
        "mutual_information": float(mi),
        "normalised_mutual_information": float(nmi),
        "implementation_note": (
            "Pearson/Spearman on numeric values; MI/NMI on 5-quantile "
            "discretisation for numeric variables; high_price left binary."
        ),
    })

corr_results = pd.DataFrame(rows)
display(corr_results)

,var_a,var_b,n,pearson,spearman,mutual_information,normalised_mutual_information,implementation_note
0,accommodates,bedrooms,14102,0.877774,0.883191,0.403374,0.385163,Pearson/Spearman on numeric values; MI/NMI on ...
1,accommodates,bathrooms,14471,0.667793,0.650323,0.257380,0.217648,Pearson/Spearman on numeric values; MI/NMI on ...
2,accommodates,distance_cbd_km,14472,0.231651,0.148605,0.052714,0.035766,Pearson/Spearman on numeric values; MI/NMI on ...
3,accommodates,amenity_count,14472,0.222542,0.236885,0.031952,0.021693,Pearson/Spearman on numeric values; MI/NMI on ...
4,accommodates,high_price,14472,0.472557,0.451785,0.106751,0.112332,Pearson/Spearman on numeric values; MI/NMI on ...
5,bedrooms,bathrooms,14101,0.730220,0.696254,0.235556,0.262694,Pearson/Spearman on numeric values; MI/NMI on ...
6,bedrooms,distance_cbd_km,14102,0.295063,0.245952,0.086805,0.073437,Pearson/Spearman on numeric values; MI/NMI on ...
7,bedrooms,amenity_count,14102,0.153830,0.179863,0.012064,0.010219,Pearson/Spearman on numeric values; MI/NMI on ...
8,bedrooms,high_price,14102,0.517891,0.496979,0.126359,0.191129,Pearson/Spearman on numeric values; MI/NMI on ...
9,bathrooms,distance_cbd_km,14471,0.175997,0.184188,0.052292,0.039672,Pearson/Spearman on numeric values; MI/NMI on ...


## 5. Target associations

This table isolates every predictor–target pair so the group can identify the strongest/weakest associations with actual values.

In [17]:
target_rows = corr_results[
    (corr_results["var_a"] == "high_price")
    | (corr_results["var_b"] == "high_price")
].copy()

target_rows["predictor"] = np.where(
    target_rows["var_a"] == "high_price",
    target_rows["var_b"],
    target_rows["var_a"],
)
target_rows["abs_pearson"] = target_rows["pearson"].abs()
target_rows["abs_spearman"] = target_rows["spearman"].abs()

display(
    target_rows[
        [
            "predictor",
            "n",
            "pearson",
            "spearman",
            "mutual_information",
            "normalised_mutual_information",
        ]
    ].sort_values("normalised_mutual_information", ascending=False)
)

,predictor,n,pearson,spearman,mutual_information,normalised_mutual_information
8,bedrooms,14102,0.517891,0.496979,0.126359,0.191129
11,bathrooms,14471,0.454505,0.452333,0.106053,0.133476
4,accommodates,14472,0.472557,0.451785,0.106751,0.112332
13,distance_cbd_km,14472,0.151772,0.158402,0.013411,0.012350
14,amenity_count,14472,0.093304,0.106652,0.010542,0.009716


## 6. Predictor–predictor relationships

This is used to identify potential redundancy/multicollinearity before modelling.

In [18]:
predictor_pairs = corr_results[
    (corr_results["var_a"] != "high_price")
    & (corr_results["var_b"] != "high_price")
].copy()

predictor_pairs["abs_pearson"] = predictor_pairs["pearson"].abs()
predictor_pairs["abs_spearman"] = predictor_pairs["spearman"].abs()

method_applicability = pd.DataFrame([
    {
        "method": "Pearson",
        "applicable": True,
        "interpretation_limit": (
            "All analysed columns are numeric; with binary high_price this is "
            "the point-biserial correlation. Sensitive to non-linearity and outliers."
        ),
    },
    {
        "method": "Spearman",
        "applicable": True,
        "interpretation_limit": (
            "Applicable to monotonic numeric relationships and less sensitive to outliers; "
            "ties are common in count and binary variables."
        ),
    },
    {
        "method": "Mutual Information",
        "applicable": True,
        "interpretation_limit": (
            "Applied after five-quantile discretisation of numeric variables; "
            "magnitude depends on binning and is not on the correlation scale."
        ),
    },
    {
        "method": "Normalised Mutual Information",
        "applicable": True,
        "interpretation_limit": (
            "Applied to the same discretised categories; useful for relative comparison "
            "but still sensitive to binning and ties."
        ),
    },
])

high_correlation_decisions = predictor_pairs.loc[
    (predictor_pairs["abs_pearson"] >= 0.80)
    | (predictor_pairs["abs_spearman"] >= 0.80),
    ["var_a", "var_b", "pearson", "spearman", "n"],
].copy()
high_correlation_decisions["analysis_role"] = (
    "Post-hoc held-out sensitivity analysis only; not used for model selection."
)
high_correlation_decisions["downstream_decision"] = (
    "Retain both in the prespecified size benchmark, then quantify held-out "
    "macro-F1 sensitivity after dropping each member of the pair."
)

display(
    predictor_pairs.sort_values(
        ["abs_pearson", "normalised_mutual_information"],
        ascending=False,
    )
)
distance_association_rows = []
for predictor, representation in [
    ("latitude", "before: raw coordinate"),
    ("longitude", "before: raw coordinate"),
    ("distance_cbd_km", "after: derived CBD distance"),
]:
    pair = df[[predictor, "high_price"]].dropna()
    distance_association_rows.append({
        "predictor": predictor,
        "representation": representation,
        "n": len(pair),
        "pearson_with_high_price": float(pearsonr(pair[predictor], pair["high_price"]).statistic),
        "spearman_with_high_price": float(spearmanr(pair[predictor], pair["high_price"]).statistic),
    })

distance_target_association = pd.DataFrame(distance_association_rows)
distance_target_association["analysis_scope"] = (
    "All eligible rows; exploratory association only, not model selection."
)

display(method_applicability)
display(distance_target_association)
display(high_correlation_decisions)


,var_a,var_b,n,pearson,spearman,mutual_information,normalised_mutual_information,implementation_note,abs_pearson,abs_spearman
0,accommodates,bedrooms,14102,0.877774,0.883191,0.403374,0.385163,Pearson/Spearman on numeric values; MI/NMI on ...,0.877774,0.883191
5,bedrooms,bathrooms,14101,0.730220,0.696254,0.235556,0.262694,Pearson/Spearman on numeric values; MI/NMI on ...,0.730220,0.696254
1,accommodates,bathrooms,14471,0.667793,0.650323,0.257380,0.217648,Pearson/Spearman on numeric values; MI/NMI on ...,0.667793,0.650323
6,bedrooms,distance_cbd_km,14102,0.295063,0.245952,0.086805,0.073437,Pearson/Spearman on numeric values; MI/NMI on ...,0.295063,0.245952
2,accommodates,distance_cbd_km,14472,0.231651,0.148605,0.052714,0.035766,Pearson/Spearman on numeric values; MI/NMI on ...,0.231651,0.148605
3,accommodates,amenity_count,14472,0.222542,0.236885,0.031952,0.021693,Pearson/Spearman on numeric values; MI/NMI on ...,0.222542,0.236885
9,bathrooms,distance_cbd_km,14471,0.175997,0.184188,0.052292,0.039672,Pearson/Spearman on numeric values; MI/NMI on ...,0.175997,0.184188
7,bedrooms,amenity_count,14102,0.153830,0.179863,0.012064,0.010219,Pearson/Spearman on numeric values; MI/NMI on ...,0.153830,0.179863
10,bathrooms,amenity_count,14471,0.128707,0.153770,0.014592,0.011078,Pearson/Spearman on numeric values; MI/NMI on ...,0.128707,0.153770
12,distance_cbd_km,amenity_count,14472,0.028157,0.005708,0.006373,0.003962,Pearson/Spearman on numeric values; MI/NMI on ...,0.028157,0.005708


,method,applicable,interpretation_limit
0,Pearson,True,All analysed columns are numeric; with binary ...
1,Spearman,True,Applicable to monotonic numeric relationships ...
2,Mutual Information,True,Applied after five-quantile discretisation of ...
3,Normalised Mutual Information,True,Applied to the same discretised categories; us...


,predictor,representation,n,pearson_with_high_price,spearman_with_high_price,analysis_scope
0,latitude,before: raw coordinate,14472,-0.023395,-0.044620,All eligible rows; exploratory association onl...
1,longitude,before: raw coordinate,14472,0.101116,0.087377,All eligible rows; exploratory association onl...
2,distance_cbd_km,after: derived CBD distance,14472,0.151772,0.158402,All eligible rows; exploratory association onl...


,var_a,var_b,pearson,spearman,n,analysis_role,downstream_decision
0,accommodates,bedrooms,0.877774,0.883191,14102,Post-hoc held-out sensitivity analysis only; n...,Retain both in the prespecified size benchmark...


## 7. Correlation matrices and figures

In [19]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

corr_results.to_csv(TABLE_OUT / "correlation_results.csv", index=False)
target_rows.to_csv(TABLE_OUT / "correlation_target_associations.csv", index=False)
predictor_pairs.to_csv(TABLE_OUT / "correlation_predictor_pairs.csv", index=False)
method_applicability.to_csv(TABLE_OUT / "correlation_method_applicability.csv", index=False)
distance_target_association.to_csv(TABLE_OUT / "preprocessing_distance_target_association.csv", index=False)
high_correlation_decisions.to_csv(TABLE_OUT / "correlation_high_pair_decisions.csv", index=False)

def symmetric_matrix(results, value_col):
    matrix = pd.DataFrame(
        np.eye(len(VARIABLES)),
        index=VARIABLES,
        columns=VARIABLES,
        dtype=float,
    )

    # MI has a meaningful non-one diagonal, but the diagonal is not used
    # in pairwise interpretation. Keep 1.0 visually for consistency.
    for _, row in results.iterrows():
        matrix.loc[row["var_a"], row["var_b"]] = row[value_col]
        matrix.loc[row["var_b"], row["var_a"]] = row[value_col]
    return matrix

for method, col in {
    "pearson": "pearson",
    "spearman": "spearman",
    "mi": "mutual_information",
    "nmi": "normalised_mutual_information",
}.items():
    matrix = symmetric_matrix(corr_results, col)
    matrix.to_csv(TABLE_OUT / f"{method}_matrix.csv")

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.values, aspect="auto")
    ax.set_xticks(range(len(VARIABLES)))
    ax.set_yticks(range(len(VARIABLES)))
    ax.set_xticklabels(VARIABLES, rotation=45, ha="right")
    ax.set_yticklabels(VARIABLES)
    ax.set_title(f"{method.upper()} pairwise matrix")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_OUT / f"correlation_{method}.png", dpi=200)
    plt.close(fig)

print("Saved correlation tables to:", TABLE_OUT.relative_to(REPO_ROOT))
print("Saved correlation figures to:", FIG_OUT.relative_to(REPO_ROOT))

Saved correlation tables to: output/tables
Saved correlation figures to: output/figures


## 8. Evidence checklist for group-written interpretation

Use the generated tables to write the report yourselves. The reproducible evidence records:
- why each method is applicable and its dataset-specific limitation;
- that the correlation analysis uses all eligible rows, including the held-out split, as a descriptive/exploratory analysis only;
- raw latitude/longitude versus derived CBD-distance associations with `high_price`;
- the strongest, weakest and near-absent predictor–target relationships;
- the strong `accommodates`–`bedrooms` relationship and a separately labelled post-hoc held-out drop-one sensitivity check;
- plausible confounders/biases to discuss using association rather than causal language.

Neither the all-row correlation table nor the held-out sensitivity check is used to select model hyperparameters; those are selected from training-only cross-validation.


# Part 3 — Supervised Learning & Evaluation

## 1. Imports and in-memory preprocessing result

In [20]:
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

df = processed.copy()
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target by split:")
display(pd.crosstab(df["split"], df["high_price"], margins=True))

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}
Target by split:


high_price,0,1,All
split,,,
test,2171,724,2895
train,8683,2894,11577
All,10854,3618,14472


## 2. Feature sets

In [21]:
SIZE_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
]

LOCATION_FEATURES = [
    "distance_cbd_km",
]

AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
]

FEATURE_SETS = {
    "size_only": SIZE_FEATURES,
    "size_location_amenities": (
        SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES
    ),
}

TARGET = "high_price"

for name, features in FEATURE_SETS.items():
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

train_df = (
    df.loc[df["split"].eq("train")].sort_values("id", kind="stable").reset_index(drop=True)
)
test_df = (
    df.loc[df["split"].eq("test")].sort_values("id", kind="stable").reset_index(drop=True)
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train positive rate:", round(train_df[TARGET].mean(), 4))
print("Test positive rate:", round(test_df[TARGET].mean(), 4))

runtime_environment = pd.DataFrame([{
    "operating_system": platform.system(),
    "os_release": platform.release(),
    "machine": platform.machine(),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scipy_version": scipy.__version__,
    "scikit_learn_version": sklearn.__version__,
    "knn_training_order": "listing id ascending (stable)",
    "knn_algorithm": "brute",
    "knn_n_jobs": 1,
}])
display(runtime_environment)


Train rows: 11577
Test rows: 2895
Train positive rate: 0.25
Test positive rate: 0.2501


,operating_system,os_release,machine,python_version,numpy_version,pandas_version,scipy_version,scikit_learn_version,knn_training_order,knn_algorithm,knn_n_jobs
0,Darwin,27.0.0,arm64,3.11.16,2.3.5,2.3.3,1.16.3,1.7.2,listing id ascending (stable),brute,1


## 3. Evaluation helpers

In [22]:
def metric_summary(y_true, y_pred, y_score=None):
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    result = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "precision_class_0": float(precision[0]),
        "recall_class_0": float(recall[0]),
        "f1_class_0": float(f1[0]),
        "support_class_0": int(support[0]),
        "precision_class_1": float(precision[1]),
        "recall_class_1": float(recall[1]),
        "f1_class_1": float(f1[1]),
        "support_class_1": int(support[1]),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, y_score))
    else:
        result["roc_auc"] = np.nan

    return result


def bootstrap_macro_f1_ci(
    y_true,
    y_pred,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores.append(
            f1_score(
                y_true[idx],
                y_pred[idx],
                average="macro",
                zero_division=0,
            )
        )

    scores = np.asarray(scores)
    lo, hi = np.quantile(scores, [alpha/2, 1-alpha/2])

    return {
        "bootstrap_mean_macro_f1": float(scores.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "n_boot": int(n_boot),
    }

## 4. Majority-class baseline

The baseline is evaluated on the held-out test set with the same metrics as the trained models.

In [23]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(
    np.zeros((len(train_df), 1)),
    train_df[TARGET],
)
baseline_pred = baseline.predict(np.zeros((len(test_df), 1)))
baseline_score = baseline.predict_proba(np.zeros((len(test_df), 1)))[:, 1]

baseline_metrics = metric_summary(
    test_df[TARGET].to_numpy(),
    baseline_pred,
    baseline_score,
)
baseline_metrics

{'accuracy': 0.7499136442141624,
 'macro_f1': 0.42854322937228584,
 'precision_class_0': 0.7499136442141624,
 'recall_class_0': 1.0,
 'f1_class_0': 0.8570864587445717,
 'support_class_0': 2171,
 'precision_class_1': 0.0,
 'recall_class_1': 0.0,
 'f1_class_1': 0.0,
 'support_class_1': 724,
 'roc_auc': 0.5}

## 5. Model pipelines and tuning grids

KNN uses median imputation + standardisation because it is distance-based.

Decision Tree uses median imputation but no scaling.

Both use 5-fold stratified CV on the training set and tune macro-F1. Every tried parameter combination is saved.

In [24]:
def make_knn_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        # Brute-force single-thread neighbour search plus stable ID ordering
        # makes exact-distance tie handling reproducible across tested runtimes.
        ("model", KNeighborsClassifier(algorithm="brute", n_jobs=1)),
    ])

def make_tree_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])

KNN_GRID = {
    "model__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

TREE_GRID = {
    "model__max_depth": [None, 3, 5, 8, 12],
    "model__min_samples_leaf": [1, 5, 15, 30],
}

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("KNN combinations:", np.prod([len(v) for v in KNN_GRID.values()]))
print("Tree combinations:", np.prod([len(v) for v in TREE_GRID.values()]))


KNN combinations: 28
Tree combinations: 20


## 6. Tune and evaluate each model × feature-set combination

In [25]:
def run_experiment(model_name, feature_set_name):
    features = FEATURE_SETS[feature_set_name]

    X_train = train_df[features]
    y_train = train_df[TARGET]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    if model_name == "KNN":
        pipeline = make_knn_pipeline()
        grid = KNN_GRID
    elif model_name == "DecisionTree":
        pipeline = make_tree_pipeline()
        grid = TREE_GRID
    else:
        raise ValueError(model_name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        scoring="f1_macro",
        cv=CV,
        n_jobs=1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    pred = search.predict(X_test)
    proba = search.predict_proba(X_test)[:, 1]

    metrics = metric_summary(y_test.to_numpy(), pred, proba)
    metrics.update({
        "model": model_name,
        "feature_set": feature_set_name,
        "best_cv_macro_f1": float(search.best_score_),
        "best_params": json.dumps(search.best_params_, sort_keys=True),
        "absolute_macro_f1_improvement_vs_baseline": (
            metrics["macro_f1"] - baseline_metrics["macro_f1"]
        ),
    })

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.insert(0, "model", model_name)
    cv_results.insert(1, "feature_set", feature_set_name)

    return {
        "search": search,
        "pred": pred,
        "proba": proba,
        "metrics": metrics,
        "cv_results": cv_results,
        "confusion_matrix": confusion_matrix(y_test, pred, labels=[0, 1]),
        "features": features,
    }

experiments = {}

for model_name in ["KNN", "DecisionTree"]:
    for feature_set_name in FEATURE_SETS:
        print("Running:", model_name, feature_set_name)
        experiments[(model_name, feature_set_name)] = run_experiment(
            model_name,
            feature_set_name,
        )

model_summary = pd.DataFrame(
    [result["metrics"] for result in experiments.values()]
)

display(
    model_summary[
        [
            "model",
            "feature_set",
            "best_cv_macro_f1",
            "accuracy",
            "macro_f1",
            "roc_auc",
            "f1_class_0",
            "f1_class_1",
            "absolute_macro_f1_improvement_vs_baseline",
            "best_params",
        ]
    ].sort_values(["model", "feature_set"])
)

Running: KNN size_only


Running: KNN size_location_amenities


Running: DecisionTree size_only


Running: DecisionTree size_location_amenities


,model,feature_set,best_cv_macro_f1,accuracy,macro_f1,roc_auc,f1_class_0,f1_class_1,absolute_macro_f1_improvement_vs_baseline,best_params
3,DecisionTree,size_location_amenities,0.753867,0.830743,0.757502,0.837520,0.890771,0.624233,0.328959,"{""model__max_depth"": 5, ""model__min_samples_le..."
2,DecisionTree,size_only,0.754895,0.824180,0.757990,0.820028,0.884554,0.631427,0.329447,"{""model__max_depth"": 3, ""model__min_samples_le..."
1,KNN,size_location_amenities,0.743824,0.817617,0.732198,0.817742,0.883444,0.580952,0.303655,"{""model__n_neighbors"": 15, ""model__p"": 1, ""mod..."
0,KNN,size_only,0.746307,0.821762,0.745464,0.815759,0.884821,0.606107,0.316921,"{""model__n_neighbors"": 21, ""model__p"": 1, ""mod..."


## 7. Hyperparameter defaults, chosen values, and specific CV effect

For each tuned parameter, this table compares the selected setting with that parameter reset to its scikit-learn default while holding the other selected parameters fixed. The difference in mean 5-fold CV macro-F1 gives a dataset-specific effect for every changed hyperparameter.

In [26]:
MODEL_DEFAULTS = {
    "KNN": {
        "model__n_neighbors": 5,
        "model__weights": "uniform",
        "model__p": 2,
    },
    "DecisionTree": {
        "model__max_depth": None,
        "model__min_samples_leaf": 1,
    },
}

hyperparameter_effect_rows = []

for (model_name, feature_set_name), exp in experiments.items():
    best_params = exp["search"].best_params_
    best_score = float(exp["search"].best_score_)
    cv_table = exp["cv_results"]

    for parameter, default_value in MODEL_DEFAULTS[model_name].items():
        chosen_value = best_params[parameter]
        changed = chosen_value != default_value

        comparison_params = dict(best_params)
        comparison_params[parameter] = default_value

        mask = cv_table["params"].apply(
            lambda p: all(p.get(k) == v for k, v in comparison_params.items())
        )

        if mask.any():
            comparison_score = float(
                cv_table.loc[mask, "mean_test_score"].iloc[0]
            )
            effect = best_score - comparison_score
        else:
            comparison_score = np.nan
            effect = np.nan

        hyperparameter_effect_rows.append({
            "model": model_name,
            "feature_set": feature_set_name,
            "parameter": parameter.replace("model__", ""),
            "default_value": str(default_value),
            "chosen_value": str(chosen_value),
            "changed_from_default": bool(changed),
            "chosen_cv_macro_f1": best_score,
            "cv_macro_f1_with_parameter_at_default": comparison_score,
            "specific_cv_macro_f1_effect": effect,
        })

hyperparameter_effects = pd.DataFrame(hyperparameter_effect_rows)
display(hyperparameter_effects)


,model,feature_set,parameter,default_value,chosen_value,changed_from_default,chosen_cv_macro_f1,cv_macro_f1_with_parameter_at_default,specific_cv_macro_f1_effect
0,KNN,size_only,n_neighbors,5,21,True,0.746307,0.713991,0.032316
1,KNN,size_only,weights,uniform,uniform,False,0.746307,0.746307,0.000000
2,KNN,size_only,p,2,1,True,0.746307,0.744768,0.001539
3,KNN,size_location_amenities,n_neighbors,5,15,True,0.743824,0.724018,0.019806
4,KNN,size_location_amenities,weights,uniform,distance,True,0.743824,0.739942,0.003882
5,KNN,size_location_amenities,p,2,1,True,0.743824,0.734143,0.009681
6,DecisionTree,size_only,max_depth,None,3,True,0.754895,0.743315,0.011580
7,DecisionTree,size_only,min_samples_leaf,1,15,True,0.754895,0.754637,0.000258
8,DecisionTree,size_location_amenities,max_depth,None,5,True,0.753867,0.739931,0.013937
9,DecisionTree,size_location_amenities,min_samples_leaf,1,30,True,0.753867,0.751949,0.001918


## 8. Incremental value of location + amenities

This table directly answers the first part of the RQ for each mandatory model by subtracting size-only performance from full-feature performance.

In [27]:
incremental_rows = []

for model_name in ["KNN", "DecisionTree"]:
    size_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_only")
    ].iloc[0]
    full_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_location_amenities")
    ].iloc[0]

    incremental_rows.append({
        "model": model_name,
        "size_only_macro_f1": float(size_row["macro_f1"]),
        "full_macro_f1": float(full_row["macro_f1"]),
        "absolute_macro_f1_change": float(
            full_row["macro_f1"] - size_row["macro_f1"]
        ),
        "size_only_accuracy": float(size_row["accuracy"]),
        "full_accuracy": float(full_row["accuracy"]),
        "absolute_accuracy_change": float(
            full_row["accuracy"] - size_row["accuracy"]
        ),
        "size_only_roc_auc": float(size_row["roc_auc"]),
        "full_roc_auc": float(full_row["roc_auc"]),
        "absolute_roc_auc_change": float(
            full_row["roc_auc"] - size_row["roc_auc"]
        ),
    })

incremental_results = pd.DataFrame(incremental_rows)
display(incremental_results)

# Follow up the strong accommodates–bedrooms correlation with an actual
# held-out sensitivity check while retaining the prespecified benchmark.
redundancy_rows = []
for model_name in ["KNN", "DecisionTree"]:
    base_exp = experiments[(model_name, "size_only")]
    base_score = base_exp["metrics"]["macro_f1"]
    for dropped in ["accommodates", "bedrooms"]:
        kept = [f for f in SIZE_FEATURES if f != dropped]
        estimator = clone(base_exp["search"].best_estimator_)
        estimator.fit(train_df[kept], train_df[TARGET])
        pred = estimator.predict(test_df[kept])
        score = f1_score(test_df[TARGET], pred, average="macro", zero_division=0)
        redundancy_rows.append({
            "model": model_name,
            "dropped_feature": dropped,
            "retained_features": " | ".join(kept),
            "original_size_only_macro_f1": float(base_score),
            "drop_one_macro_f1": float(score),
            "macro_f1_change_vs_original": float(score - base_score),
        })

redundancy_sensitivity = pd.DataFrame(redundancy_rows)
redundancy_sensitivity["analysis_role"] = (
    "Post-hoc held-out sensitivity only; not used for model selection."
)
display(redundancy_sensitivity)

# Quantify the exact-distance boundary ties that make KNN implementation
# details important for this low-cardinality size-only feature set.
size_imputer = SimpleImputer(strategy="median")
size_scaler = StandardScaler()
z_train = size_scaler.fit_transform(size_imputer.fit_transform(train_df[SIZE_FEATURES]))
z_test = size_scaler.transform(size_imputer.transform(test_df[SIZE_FEATURES]))
unique_size_combinations = int(train_df[SIZE_FEATURES].drop_duplicates().shape[0])

knn_tie_rows = []
for p_value in KNN_GRID["model__p"]:
    for k_value in KNN_GRID["model__n_neighbors"]:
        neighbour_search = NearestNeighbors(
            n_neighbors=k_value + 1,
            metric="minkowski",
            p=p_value,
            algorithm="brute",
            n_jobs=1,
        ).fit(z_train)
        distances = neighbour_search.kneighbors(z_test, return_distance=True)[0]
        boundary_tie = np.isclose(
            distances[:, k_value - 1],
            distances[:, k_value],
            rtol=1e-12,
            atol=1e-12,
        )
        knn_tie_rows.append({
            "p": p_value,
            "k": k_value,
            "train_rows": len(train_df),
            "unique_size_feature_combinations": unique_size_combinations,
            "test_boundary_tie_n": int(boundary_tie.sum()),
            "test_boundary_tie_pct": float(boundary_tie.mean() * 100),
        })

knn_tie_diagnostics = pd.DataFrame(knn_tie_rows)
display(knn_tie_diagnostics)


# Quantify the downstream effect of deriving and adding CBD distance. The
# full-model hyperparameters are held fixed, so this is an ablation rather
# than a second model-selection exercise.
distance_ablation_rows = []
for model_name in ["KNN", "DecisionTree"]:
    full_exp = experiments[(model_name, "size_location_amenities")]
    without_distance = [
        feature for feature in full_exp["features"]
        if feature != "distance_cbd_km"
    ]
    estimator = clone(full_exp["search"].best_estimator_)
    estimator.fit(train_df[without_distance], train_df[TARGET])
    pred = estimator.predict(test_df[without_distance])
    proba = estimator.predict_proba(test_df[without_distance])[:, 1]
    no_distance_metrics = metric_summary(test_df[TARGET].to_numpy(), pred, proba)
    distance_ablation_rows.append({
        "model": model_name,
        "analysis_role": "Post-hoc held-out ablation; fixed full-model hyperparameters.",
        "with_distance_macro_f1": float(full_exp["metrics"]["macro_f1"]),
        "without_distance_macro_f1": float(no_distance_metrics["macro_f1"]),
        "macro_f1_change_from_adding_distance": float(
            full_exp["metrics"]["macro_f1"] - no_distance_metrics["macro_f1"]
        ),
        "with_distance_roc_auc": float(full_exp["metrics"]["roc_auc"]),
        "without_distance_roc_auc": float(no_distance_metrics["roc_auc"]),
        "roc_auc_change_from_adding_distance": float(
            full_exp["metrics"]["roc_auc"] - no_distance_metrics["roc_auc"]
        ),
    })

distance_ablation = pd.DataFrame(distance_ablation_rows)
display(distance_ablation)

# The stable ascending-ID order is the recorded run. Reversing only the
# training-row order demonstrates the remaining KNN sensitivity caused by
# exact-distance boundary ties; this audit does not replace the recorded run.
reverse_train_df = train_df.sort_values("id", ascending=False, kind="stable")
knn_order_sensitivity_rows = []
for feature_set_name, features in FEATURE_SETS.items():
    recorded = experiments[("KNN", feature_set_name)]
    estimator = clone(recorded["search"].best_estimator_)
    estimator.fit(reverse_train_df[features], reverse_train_df[TARGET])
    reverse_pred = estimator.predict(test_df[features])
    reverse_proba = estimator.predict_proba(test_df[features])[:, 1]
    reverse_metrics = metric_summary(
        test_df[TARGET].to_numpy(), reverse_pred, reverse_proba
    )
    knn_order_sensitivity_rows.append({
        "feature_set": feature_set_name,
        "recorded_training_order": "listing id ascending",
        "audit_training_order": "listing id descending",
        "recorded_macro_f1": float(recorded["metrics"]["macro_f1"]),
        "reverse_order_macro_f1": float(reverse_metrics["macro_f1"]),
        "macro_f1_difference_reverse_minus_recorded": float(
            reverse_metrics["macro_f1"] - recorded["metrics"]["macro_f1"]
        ),
        "recorded_roc_auc": float(recorded["metrics"]["roc_auc"]),
        "reverse_order_roc_auc": float(reverse_metrics["roc_auc"]),
        "roc_auc_difference_reverse_minus_recorded": float(
            reverse_metrics["roc_auc"] - recorded["metrics"]["roc_auc"]
        ),
        "analysis_role": "Reproducibility stress test; not model selection.",
    })

knn_order_sensitivity = pd.DataFrame(knn_order_sensitivity_rows)
display(knn_order_sensitivity)


,model,size_only_macro_f1,full_macro_f1,absolute_macro_f1_change,size_only_accuracy,full_accuracy,absolute_accuracy_change,size_only_roc_auc,full_roc_auc,absolute_roc_auc_change
0,KNN,0.745464,0.732198,-0.013266,0.821762,0.817617,-0.004145,0.815759,0.817742,0.001983
1,DecisionTree,0.757990,0.757502,-0.000488,0.824180,0.830743,0.006563,0.820028,0.837520,0.017493


,model,dropped_feature,retained_features,original_size_only_macro_f1,drop_one_macro_f1,macro_f1_change_vs_original,analysis_role
0,KNN,accommodates,bedrooms | beds | bathrooms,0.745464,0.758931,0.013466,Post-hoc held-out sensitivity only; not used f...
1,KNN,bedrooms,accommodates | beds | bathrooms,0.745464,0.739490,-0.005975,Post-hoc held-out sensitivity only; not used f...
2,DecisionTree,accommodates,bedrooms | beds | bathrooms,0.757990,0.757152,-0.000838,Post-hoc held-out sensitivity only; not used f...
3,DecisionTree,bedrooms,accommodates | beds | bathrooms,0.757990,0.735944,-0.022046,Post-hoc held-out sensitivity only; not used f...


,p,k,train_rows,unique_size_feature_combinations,test_boundary_tie_n,test_boundary_tie_pct
0,1,3,11577,867,2782,96.096718
1,1,5,11577,867,2787,96.269430
2,1,7,11577,867,2764,95.474957
3,1,11,11577,867,2800,96.718480
4,1,15,11577,867,2815,97.236615
5,1,21,11577,867,2799,96.683938
6,1,31,11577,867,2812,97.132988
7,2,3,11577,867,2778,95.958549
8,2,5,11577,867,2779,95.993092
9,2,7,11577,867,2772,95.751295


,model,analysis_role,with_distance_macro_f1,without_distance_macro_f1,macro_f1_change_from_adding_distance,with_distance_roc_auc,without_distance_roc_auc,roc_auc_change_from_adding_distance
0,KNN,Post-hoc held-out ablation; fixed full-model h...,0.732198,0.721263,0.010935,0.817742,0.774061,0.043682
1,DecisionTree,Post-hoc held-out ablation; fixed full-model h...,0.757502,0.747990,0.009513,0.837520,0.827440,0.010080


,feature_set,recorded_training_order,audit_training_order,recorded_macro_f1,reverse_order_macro_f1,macro_f1_difference_reverse_minus_recorded,recorded_roc_auc,reverse_order_roc_auc,roc_auc_difference_reverse_minus_recorded,analysis_role
0,size_only,listing id ascending,listing id descending,0.745464,0.742879,-0.002585,0.815759,0.813331,-0.002428,Reproducibility stress test; not model selection.
1,size_location_amenities,listing id ascending,listing id descending,0.732198,0.732198,0.000000,0.817742,0.817716,-0.000027,Reproducibility stress test; not model selection.


## 9. Uncertainty quantification

The full-feature model used for its standalone macro-F1 interval is chosen by training-only cross-validation, not by the held-out test score. Paired bootstraps on the same held-out rows estimate both:

1. the macro-F1 difference between the full and size-only feature sets; and
2. the ROC-AUC difference between the same two feature sets.

These intervals directly quantify uncertainty in the research-question comparison. They are evaluation evidence, not additional model-selection criteria.


In [28]:
full_results = {
    model: experiments[(model, "size_location_amenities")]
    for model in ["KNN", "DecisionTree"]
}

uncertainty_model = max(
    full_results,
    key=lambda m: full_results[m]["metrics"]["best_cv_macro_f1"],
)
uncertainty_exp = full_results[uncertainty_model]

uncertainty = bootstrap_macro_f1_ci(
    test_df[TARGET].to_numpy(),
    uncertainty_exp["pred"],
    n_boot=2000,
    random_state=RANDOM_STATE,
)
uncertainty["model"] = uncertainty_model
uncertainty["feature_set"] = "size_location_amenities"

def paired_bootstrap_macro_f1_difference(
    y_true,
    pred_full,
    pred_size,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    pred_full = np.asarray(pred_full)
    pred_size = np.asarray(pred_size)
    observed = (
        f1_score(y_true, pred_full, average="macro", zero_division=0)
        - f1_score(y_true, pred_size, average="macro", zero_division=0)
    )
    rng = np.random.default_rng(random_state)
    differences = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), size=len(y_true))
        differences.append(
            f1_score(y_true[idx], pred_full[idx], average="macro", zero_division=0)
            - f1_score(y_true[idx], pred_size[idx], average="macro", zero_division=0)
        )
    differences = np.asarray(differences)
    lo, hi = np.quantile(differences, [alpha / 2, 1 - alpha / 2])
    return {
        "observed_macro_f1_difference_full_minus_size": float(observed),
        "bootstrap_mean_difference": float(differences.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "bootstrap_probability_full_greater_than_size": float((differences > 0).mean()),
        "n_boot": int(n_boot),
    }

paired_rows = []
for offset, model_name in enumerate(["KNN", "DecisionTree"]):
    result = paired_bootstrap_macro_f1_difference(
        test_df[TARGET].to_numpy(),
        experiments[(model_name, "size_location_amenities")]["pred"],
        experiments[(model_name, "size_only")]["pred"],
        n_boot=2000,
        random_state=RANDOM_STATE + offset,
    )
    result["model"] = model_name
    paired_rows.append(result)

paired_incremental_uncertainty = pd.DataFrame(paired_rows)
display(pd.DataFrame([uncertainty]))
display(paired_incremental_uncertainty)


def paired_bootstrap_roc_auc_difference(
    y_true,
    score_full,
    score_size,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    score_full = np.asarray(score_full)
    score_size = np.asarray(score_size)
    observed = roc_auc_score(y_true, score_full) - roc_auc_score(y_true, score_size)
    rng = np.random.default_rng(random_state)
    differences = []
    while len(differences) < n_boot:
        idx = rng.integers(0, len(y_true), size=len(y_true))
        if np.unique(y_true[idx]).size < 2:
            continue
        differences.append(
            roc_auc_score(y_true[idx], score_full[idx])
            - roc_auc_score(y_true[idx], score_size[idx])
        )
    differences = np.asarray(differences)
    lo, hi = np.quantile(differences, [alpha / 2, 1 - alpha / 2])
    return {
        "observed_roc_auc_difference_full_minus_size": float(observed),
        "bootstrap_mean_difference": float(differences.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "bootstrap_probability_full_greater_than_size": float((differences > 0).mean()),
        "n_boot": int(n_boot),
    }

paired_auc_rows = []
for offset, model_name in enumerate(["KNN", "DecisionTree"]):
    result = paired_bootstrap_roc_auc_difference(
        test_df[TARGET].to_numpy(),
        experiments[(model_name, "size_location_amenities")]["proba"],
        experiments[(model_name, "size_only")]["proba"],
        n_boot=2000,
        random_state=RANDOM_STATE + 100 + offset,
    )
    result["model"] = model_name
    paired_auc_rows.append(result)

paired_auc_uncertainty = pd.DataFrame(paired_auc_rows)
display(paired_auc_uncertainty)


,bootstrap_mean_macro_f1,ci_lower_95,ci_upper_95,n_boot,model,feature_set
0,0.757433,0.737865,0.777451,2000,DecisionTree,size_location_amenities


,observed_macro_f1_difference_full_minus_size,bootstrap_mean_difference,ci_lower_95,ci_upper_95,bootstrap_probability_full_greater_than_size,n_boot,model
0,-0.013266,-0.012992,-0.029556,0.004003,0.0670,2000,KNN
1,-0.000488,-0.000460,-0.011464,0.010865,0.4595,2000,DecisionTree


,observed_roc_auc_difference_full_minus_size,bootstrap_mean_difference,ci_lower_95,ci_upper_95,bootstrap_probability_full_greater_than_size,n_boot,model
0,0.001983,0.001707,-0.013118,0.016376,0.598,2000,KNN
1,0.017493,0.017624,0.010455,0.025160,1.000,2000,DecisionTree


## 10. Feature influence for both models

Permutation importance is calculated on the same held-out test set for the two full-feature models, using macro-F1. This gives feature-level influence in the original feature space for both KNN and Decision Tree.

In [29]:
importance_rows = []

for model_name in ["KNN", "DecisionTree"]:
    exp = experiments[(model_name, "size_location_amenities")]
    features = exp["features"]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    perm = permutation_importance(
        exp["search"].best_estimator_,
        X_test,
        y_test,
        scoring="f1_macro",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    for feature, mean_imp, std_imp in zip(
        features,
        perm.importances_mean,
        perm.importances_std,
    ):
        importance_rows.append({
            "model": model_name,
            "feature": feature,
            "permutation_importance_mean": float(mean_imp),
            "permutation_importance_std": float(std_imp),
        })

permutation_importance_table = pd.DataFrame(importance_rows)
display(
    permutation_importance_table.sort_values(
        ["model", "permutation_importance_mean"],
        ascending=[True, False],
    )
)

,model,feature,permutation_importance_mean,permutation_importance_std
13,DecisionTree,bedrooms,0.240683,0.008897
12,DecisionTree,accommodates,0.033180,0.003169
16,DecisionTree,distance_cbd_km,0.016930,0.005667
15,DecisionTree,bathrooms,0.004118,0.002488
17,DecisionTree,amenity_count,0.003953,0.002484
14,DecisionTree,beds,0.000000,0.000000
18,DecisionTree,has_pool,0.000000,0.000000
19,DecisionTree,has_free_parking,0.000000,0.000000
20,DecisionTree,has_air_conditioning,0.000000,0.000000
21,DecisionTree,has_dedicated_workspace,0.000000,0.000000


## 11. Save evaluation outputs and figures

In [30]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

baseline_table = pd.DataFrame([{
    "model": "MajorityClassBaseline",
    "feature_set": "none",
    **baseline_metrics,
}])

baseline_table.to_csv(TABLE_OUT / "baseline_metrics.csv", index=False)
model_summary.to_csv(TABLE_OUT / "model_summary.csv", index=False)
hyperparameter_effects.to_csv(TABLE_OUT / "hyperparameter_effects.csv", index=False)
incremental_results.to_csv(TABLE_OUT / "model_incremental_value.csv", index=False)
paired_incremental_uncertainty.to_csv(TABLE_OUT / "model_incremental_bootstrap.csv", index=False)
paired_auc_uncertainty.to_csv(TABLE_OUT / "model_auc_incremental_bootstrap.csv", index=False)
distance_ablation.to_csv(TABLE_OUT / "model_distance_ablation.csv", index=False)
knn_order_sensitivity.to_csv(TABLE_OUT / "knn_order_sensitivity.csv", index=False)
runtime_environment.to_csv(TABLE_OUT / "runtime_environment.csv", index=False)
redundancy_sensitivity.to_csv(TABLE_OUT / "model_redundancy_sensitivity.csv", index=False)
knn_tie_diagnostics.to_csv(TABLE_OUT / "knn_tie_diagnostics.csv", index=False)
permutation_importance_table.to_csv(
    TABLE_OUT / "model_permutation_importance.csv",
    index=False,
)
pd.DataFrame([uncertainty]).to_csv(
    TABLE_OUT / "model_uncertainty.csv",
    index=False,
)

for (model_name, feature_set_name), exp in experiments.items():
    safe_model = model_name.lower()
    safe_set = feature_set_name.lower()

    keep_cols = [
        c for c in exp["cv_results"].columns
        if c.startswith("param_")
        or c in [
            "model",
            "feature_set",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "rank_test_score",
            "params",
        ]
    ]
    exp["cv_results"][keep_cols].to_csv(
        TABLE_OUT / f"cv_{safe_model}_{safe_set}.csv",
        index=False,
    )

    cm = pd.DataFrame(
        exp["confusion_matrix"],
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
    cm.to_csv(
        TABLE_OUT / f"confusion_{safe_model}_{safe_set}.csv"
    )

plot_df = model_summary.pivot(
    index="model",
    columns="feature_set",
    values="macro_f1",
)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(plot_df.index))
width = 0.36

ax.bar(
    x - width/2,
    plot_df["size_only"],
    width,
    label="Size only",
)
ax.bar(
    x + width/2,
    plot_df["size_location_amenities"],
    width,
    label="Size + location + amenities",
)
ax.axhline(
    baseline_metrics["macro_f1"],
    linestyle="--",
    linewidth=1.2,
    label="Majority baseline",
)
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("Held-out macro-F1")
ax.set_title("Model performance by feature set")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_OUT / "model_macro_f1_comparison.png", dpi=200)
plt.close(fig)

for model_name in ["KNN", "DecisionTree"]:
    imp = permutation_importance_table[
        permutation_importance_table["model"] == model_name
    ].sort_values("permutation_importance_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(
        imp["feature"],
        imp["permutation_importance_mean"],
        xerr=imp["permutation_importance_std"],
    )
    ax.set_xlabel("Permutation importance (macro-F1 decrease)")
    ax.set_title(f"{model_name} feature influence")
    fig.tight_layout()
    fig.savefig(
        FIG_OUT / f"permutation_importance_{model_name.lower()}.png",
        dpi=200,
    )
    plt.close(fig)

print("Saved model outputs to:", TABLE_OUT.relative_to(REPO_ROOT))
print("Saved model figures to:", FIG_OUT.relative_to(REPO_ROOT))

Saved model outputs to: output/tables
Saved model figures to: output/figures


## 12. Hyperparameter and reproducibility evidence checklist

For the group-written Methodology/Discussion, use the saved tables to state:
- every value tried, each tuned parameter's default, selected value, and observed validation-score effect;
- the exact recorded operating system, architecture, Python, and package versions;
- that KNN uses stable listing-ID ordering and brute-force single-thread neighbour search;
- the measured frequency of exact-distance boundary ties and the training-order stress test as KNN limitations;
- the paired-bootstrap intervals for full-minus-size-only macro-F1 and ROC-AUC;
- the post-hoc distance ablation and correlated-size-feature drop-one checks, clearly labelled as held-out sensitivity analyses rather than model selection.

Use the actual output tables rather than generic model descriptions.


# Part 4 — Feature Selection

## 1. Imports and in-memory modelling results

In [31]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

df = processed.copy()
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}


## 2. Use the exact modelling feature set and training rows

In [32]:
FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
]

TARGET = "high_price"
ID_COL = "id"

missing = [c for c in FEATURES + [TARGET, ID_COL, "split"] if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

train_df = (
    df.loc[df["split"].eq("train")].sort_values("id", kind="stable").reset_index(drop=True)
)
test_df = (
    df.loc[df["split"].eq("test")].sort_values("id", kind="stable").reset_index(drop=True)
)

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int)

print("Training rows:", len(train_df))
print("Target counts:", y_train.value_counts().sort_index().to_dict())

Training rows: 11577
Target counts: {0: 8683, 1: 2894}


## 3. Embedded method — tuned Decision Tree feature importance

The best hyperparameters selected for the full-feature Decision Tree in the modelling stage are reused here. This keeps the embedded ranking traceable to the model comparison rather than fitting an unrelated tree.


In [33]:
tree_row = model_summary.loc[
    (model_summary["model"] == "DecisionTree")
    & (model_summary["feature_set"] == "size_location_amenities")
]

if len(tree_row) != 1:
    raise ValueError("Expected exactly one full-feature DecisionTree result from modelling.")

best_params_raw = tree_row.iloc[0]["best_params"]
best_params = json.loads(best_params_raw)

# GridSearchCV stores pipeline parameter names such as model__max_depth.
tree_kwargs = {
    key.replace("model__", ""): value
    for key, value in best_params.items()
    if key.startswith("model__")
}
tree_kwargs["random_state"] = RANDOM_STATE

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index,
)

tree = DecisionTreeClassifier(**tree_kwargs)
tree.fit(X_train_imp, y_train)

embedded_scores = pd.Series(
    tree.feature_importances_,
    index=FEATURES,
    name="embedded_score",
).sort_values(ascending=False)

embedded_top3 = embedded_scores.head(3)
display(embedded_top3.to_frame())

,embedded_score
bedrooms,0.780613
distance_cbd_km,0.080485
bathrooms,0.059686


## 4. Filter method — Mutual Information

Mutual Information is calculated on the same training rows. Binary amenity indicators are declared discrete; the remaining numeric/count variables are treated as continuous. Median imputation is fitted only on training data.

In [34]:
DISCRETE_FEATURES = {
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
}

discrete_mask = np.array([f in DISCRETE_FEATURES for f in FEATURES], dtype=bool)

mi_values = mutual_info_classif(
    X_train_imp,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE,
)

filter_scores = pd.Series(
    mi_values,
    index=FEATURES,
    name="filter_mi_score",
).sort_values(ascending=False)

filter_top3 = filter_scores.head(3)
display(filter_top3.to_frame())

,filter_mi_score
bedrooms,0.142634
accommodates,0.127700
bathrooms,0.107099


## 5. Compare the two top-3 lists

The comparison table preserves both ranks and scores. The report discussion should explain any disagreement using these actual values and the different mechanics of embedded vs filter selection.

In [35]:
rank_table = pd.DataFrame({
    "feature": FEATURES,
    "embedded_score": embedded_scores.reindex(FEATURES).values,
    "filter_mi_score": filter_scores.reindex(FEATURES).values,
})

rank_table["embedded_rank"] = rank_table["embedded_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["filter_rank"] = rank_table["filter_mi_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["rank_difference"] = (
    rank_table["embedded_rank"] - rank_table["filter_rank"]
).abs()

rank_table = rank_table.sort_values(
    ["embedded_rank", "filter_rank", "feature"]
).reset_index(drop=True)

display(rank_table)

top3_comparison = pd.DataFrame({
    "embedded_feature": embedded_top3.index.tolist(),
    "embedded_score": embedded_top3.values.tolist(),
    "filter_feature": filter_top3.index.tolist(),
    "filter_score": filter_top3.values.tolist(),
})
display(top3_comparison)

,feature,embedded_score,filter_mi_score,embedded_rank,filter_rank,rank_difference
0,bedrooms,0.780613,0.142634,1,1,0
1,distance_cbd_km,0.080485,0.022122,2,6,4
2,bathrooms,0.059686,0.107099,3,3,0
3,accommodates,0.059537,0.127700,4,2,2
4,amenity_count,0.016097,0.022659,5,5,0
5,beds,0.001791,0.094425,6,4,2
6,has_free_parking,0.001791,0.012479,7,7,0
7,has_dryer,0.000000,0.004514,8,8,0
8,has_washer,0.000000,0.004036,8,9,1
9,has_dedicated_workspace,0.000000,0.001689,8,10,2


,embedded_feature,embedded_score,filter_feature,filter_score
0,bedrooms,0.780613,bedrooms,0.142634
1,distance_cbd_km,0.080485,accommodates,0.127700
2,bathrooms,0.059686,bathrooms,0.107099


## 6. Hard-case listing

A hard case should be difficult for the fitted model, not merely extreme on one feature. The procedure below evaluates the held-out test set with the tuned full-feature Decision Tree and selects the **highest-confidence misclassification**. The exported row records its true label, predicted label, prediction confidence, high-price probability, price, all model features, and its exact price margin above/below the training-Q75 decision boundary.

This is a post-model diagnostic. It is not used for training, tuning, or choosing the model.


In [36]:
X_test_imp = pd.DataFrame(
    imputer.transform(test_df[FEATURES]),
    columns=FEATURES,
    index=test_df.index,
)
test_prediction = tree.predict(X_test_imp).astype(int)
test_probability = tree.predict_proba(X_test_imp)
predicted_probability = test_probability[
    np.arange(len(test_prediction)), test_prediction
]

hard_candidates = test_df[[ID_COL, TARGET, "price_clean"] + FEATURES].copy()
hard_candidates["predicted_high_price"] = test_prediction
hard_candidates["probability_high_price"] = test_probability[:, 1]
hard_candidates["predicted_class_probability"] = predicted_probability
hard_candidates["prediction_correct"] = (
    hard_candidates[TARGET].to_numpy() == test_prediction
)
hard_candidates = hard_candidates.loc[~hard_candidates["prediction_correct"]]

if hard_candidates.empty:
    raise ValueError("The held-out test set contains no Decision Tree misclassification.")

hard_idx = hard_candidates["predicted_class_probability"].idxmax()
hard_case = hard_candidates.loc[[hard_idx]].copy()
hard_case.insert(1, "split", "test")
hard_case.insert(
    2,
    "hard_case_reason",
    "highest-confidence misclassification by tuned full-feature Decision Tree",
)
training_price_q75 = float(train_df["price_clean"].quantile(0.75))
hard_case.insert(3, "training_price_q75", training_price_q75)
hard_case.insert(
    4,
    "price_margin_from_training_q75",
    hard_case["price_clean"] - training_price_q75,
)
hard_case.insert(
    5,
    "boundary_case_note",
    "True high-price listing close to the training-Q75 threshold.",
)

display(hard_case.T)


,65
id,6520432
split,test
hard_case_reason,highest-confidence misclassification by tuned ...
training_price_q75,385.15
price_margin_from_training_q75,14.1
boundary_case_note,True high-price listing close to the training-...
high_price,1
price_clean,399.25
accommodates,2
bedrooms,1.0


## 7. Concrete opposite-ranking scenario

Zero-importance ties can create a large numerical rank gap without a meaningful substantive disagreement. The selection below therefore requires both methods to assign positive scores and looks for a feature that the filter ranks above the embedded tree. Ties are resolved toward the smaller embedded importance, highlighting a genuine redundancy/conditional-importance contrast.


In [37]:
meaningful_disagreements = rank_table.loc[
    (rank_table["embedded_score"] > 0)
    & (rank_table["filter_mi_score"] > 0)
    & (rank_table["filter_rank"] < rank_table["embedded_rank"])
].copy()
meaningful_disagreements["filter_over_embedded_rank_gap"] = (
    meaningful_disagreements["embedded_rank"]
    - meaningful_disagreements["filter_rank"]
)

if meaningful_disagreements.empty:
    raise ValueError("No positive-score filter-over-embedded disagreement found.")

largest_disagreement = meaningful_disagreements.sort_values(
    ["filter_over_embedded_rank_gap", "embedded_score", "filter_mi_score"],
    ascending=[False, True, False],
).iloc[[0]]

display(largest_disagreement)


,feature,embedded_score,filter_mi_score,embedded_rank,filter_rank,rank_difference,filter_over_embedded_rank_gap
5,beds,0.001791,0.094425,6,4,2,2


## 8. Save reproducible feature-selection outputs

In [38]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
TABLE_OUT.mkdir(parents=True, exist_ok=True)

rank_table.to_csv(TABLE_OUT / "feature_selection_rankings.csv", index=False)
top3_comparison.to_csv(TABLE_OUT / "feature_selection_top3.csv", index=False)
hard_case.to_csv(TABLE_OUT / "feature_selection_hard_case.csv", index=False)
largest_disagreement.to_csv(
    TABLE_OUT / "feature_selection_largest_rank_disagreement.csv",
    index=False,
)

print("Saved:")
for name in [
    "feature_selection_rankings.csv",
    "feature_selection_top3.csv",
    "feature_selection_hard_case.csv",
    "feature_selection_largest_rank_disagreement.csv",
]:
    print(" -", (TABLE_OUT / name).relative_to(REPO_ROOT))

Saved:
 - output/tables/feature_selection_rankings.csv
 - output/tables/feature_selection_top3.csv
 - output/tables/feature_selection_hard_case.csv
 - output/tables/feature_selection_largest_rank_disagreement.csv


## 9. Evidence checklist

Before the group-written report is finalised, verify:
- embedded top 3 + scores are quoted correctly;
- filter top 3 + scores are quoted correctly;
- disagreement is explained using the actual rank table;
- the hard-case listing ID and relevant actual values are quoted;
- the opposite-ranking scenario names a real feature from this group's feature set;
- feature-selection limitations are specific to the observed results, not generic.